# PX4 Phase 1 Closed-Loop Dynamics PINN Training v13

This notebook trains the first closed-loop Phase 1 dynamics model from the v14 PX4/Gazebo dataset.

Model contract:

```text
x_t, setpoint_t, prev_setpoint_t, dsetpoint_t, dt_s -> dx_t
```

The setpoint is the upper-controller command sent through PX4 offboard position/velocity/yaw interfaces, not raw open-loop thrust.


In [1]:
# Optional in Colab. Safe to skip locally if Drive is already mounted.
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print('Drive mount skipped:', type(exc).__name__, exc)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import json
import math
import os
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

SEED = 7
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)
print('torch:', torch.__version__)


device: cuda
torch: 2.10.0+cu128


In [3]:
NOTEBOOK_REVISION = 'v15_block_rollout'
DATASET_GLOB_CANDIDATES = [
    '/content/drive/MyDrive/**/px4_phase1_closed_loop_yawrel_block_dataset_v7_*',
    '/content/drive/MyDrive/**/processed/px4_phase1_closed_loop_yawrel_block_dataset_v7_*',
    '/content/**/px4_phase1_closed_loop_yawrel_block_dataset_v7_*',
]
MANUAL_DATASET_DIR = ''  # Set this to a processed v5 dataset directory if auto-search misses it.

BATCH_SIZE = 2048
EPOCHS = 260
LR = 2.5e-3
WEIGHT_DECAY = 2e-4
GRAD_CLIP = 2.0
DROPOUT = 0.035
HIDDEN = 256
DEPTH = 4

# Soft physics terms. Keep these modest; the closed-loop PX4 plant is not a simple rigid-body-only model.
ALT_KIN_WEIGHT = 0.08
YAW_KIN_WEIGHT = 0.03
RATE_SMOOTH_WEIGHT = 0.01


In [4]:
import glob

def find_dataset_dir():
    if MANUAL_DATASET_DIR:
        p = Path(MANUAL_DATASET_DIR)
        if (p / 'train.csv').exists():
            return p
        raise FileNotFoundError(f'MANUAL_DATASET_DIR does not contain train.csv: {p}')
    matches = []
    for pat in DATASET_GLOB_CANDIDATES:
        matches.extend(Path(p) for p in glob.glob(pat, recursive=True))
    matches = sorted(set(p for p in matches if (p / 'train.csv').exists()))
    if not matches:
        raise FileNotFoundError('No processed v7 block-split dataset found. Run px4_dataset_builder_v7_block_split first or set MANUAL_DATASET_DIR.')
    return matches[-1]

DATASET_DIR = find_dataset_dir()
print('DATASET_DIR:', DATASET_DIR)
for name in ['dataset_summary.csv', 'filter_report.csv', 'metadata.json']:
    p = DATASET_DIR / name
    print(name, p.exists())


DATASET_DIR: /content/drive/MyDrive/px4_datasets/processed/px4_phase1_closed_loop_yawrel_block_dataset_v7_20260510_073654
dataset_summary.csv True
filter_report.csv True
metadata.json True


In [5]:
train_df = pd.read_csv(DATASET_DIR / 'train.csv')
val_df = pd.read_csv(DATASET_DIR / 'val.csv')
test_df = pd.read_csv(DATASET_DIR / 'test.csv')
summary_df = pd.read_csv(DATASET_DIR / 'dataset_summary.csv')
print('rows:', {k: len(v) for k, v in [('train', train_df), ('val', val_df), ('test', test_df)]})
display(summary_df)

STATE_COLS = [
    'relative_altitude_m', 'vel_north_m_s', 'vel_east_m_s', 'vel_down_m_s',
    'roll_deg', 'pitch_deg', 'yaw_deg',
    'roll_rate_rad_s', 'pitch_rate_rad_s', 'yaw_rate_rad_s',
]
ACTION_COLS = [
    'ref_north_m', 'ref_east_m', 'ref_down_m',
    'ref_north_m_s', 'ref_east_m_s', 'ref_down_m_s', 'ref_yaw_deg', 'ref_yaw_offset_deg',
]
TARGET_COLS = [f'dx_{c}' for c in STATE_COLS]
print('target columns:', TARGET_COLS)


rows: {'train': 23782, 'val': 4848, 'test': 5529}


,split,scenario,samples
0,test,C00_position_hold_25m,480
1,test,C01_altitude_step_25_28_25m,936
2,test,C02_altitude_step_25_22_25m,772
3,test,C03_north_position_step_pm2m,419
4,test,C04_east_position_step_pm2m,407
5,test,C05_yaw_position_hold_pm20deg,640
6,test,C06_velocity_north_doublet_0p5mps,271
7,test,C07_velocity_east_doublet_0p5mps,160
8,test,C08_vertical_velocity_doublet_0p25mps,198
9,test,C09_mixed_position_sequence,1246


target columns: ['dx_relative_altitude_m', 'dx_vel_north_m_s', 'dx_vel_east_m_s', 'dx_vel_down_m_s', 'dx_roll_deg', 'dx_pitch_deg', 'dx_yaw_deg', 'dx_roll_rate_rad_s', 'dx_pitch_rate_rad_s', 'dx_yaw_rate_rad_s']


In [6]:
def angle_sin_deg(s):
    return np.sin(np.deg2rad(pd.to_numeric(s, errors='coerce').to_numpy(dtype=np.float32)))

def angle_cos_deg(s):
    return np.cos(np.deg2rad(pd.to_numeric(s, errors='coerce').to_numpy(dtype=np.float32)))

def make_features(df):
    parts = []
    names = []

    # Current state. Use sin/cos for yaw to avoid discontinuity at +/-180 deg.
    for col in STATE_COLS:
        src = f'x_{col}'
        if col == 'yaw_deg':
            parts.append(angle_sin_deg(df[src])[:, None]); names.append('x_yaw_sin')
            parts.append(angle_cos_deg(df[src])[:, None]); names.append('x_yaw_cos')
        else:
            parts.append(pd.to_numeric(df[src], errors='coerce').to_numpy(dtype=np.float32)[:, None]); names.append(src)

    # Current, previous, and delta setpoints. Encode yaw references cyclically.
    for prefix in ['u_', 'prev_u_', 'du_']:
        for col in ACTION_COLS:
            src = prefix + col
            if col == 'ref_yaw_deg' and prefix != 'du_':
                parts.append(angle_sin_deg(df[src])[:, None]); names.append(src + '_sin')
                parts.append(angle_cos_deg(df[src])[:, None]); names.append(src + '_cos')
            else:
                values = pd.to_numeric(df[src], errors='coerce').to_numpy(dtype=np.float32)
                if col == 'ref_yaw_deg':
                    values = ((values + 180.0) % 360.0) - 180.0
                parts.append(values[:, None]); names.append(src)

    parts.append(pd.to_numeric(df['dt_s'], errors='coerce').to_numpy(dtype=np.float32)[:, None]); names.append('dt_s')
    X = np.concatenate(parts, axis=1).astype(np.float32)
    return X, names

def make_targets(df):
    Y = np.stack([pd.to_numeric(df[c], errors='coerce').to_numpy(dtype=np.float32) for c in TARGET_COLS], axis=1)
    return Y.astype(np.float32)

X_train, FEATURE_COLS = make_features(train_df)
X_val, _ = make_features(val_df)
X_test, _ = make_features(test_df)
Y_train = make_targets(train_df)
Y_val = make_targets(val_df)
Y_test = make_targets(test_df)

print('feature dim:', X_train.shape[1])
print('target dim:', Y_train.shape[1])
print('bad train finite:', np.isfinite(X_train).all(), np.isfinite(Y_train).all())


feature dim: 38
target dim: 10
bad train finite: True True


In [7]:
class Standardizer:
    def __init__(self, x, eps=1e-6):
        self.mean = torch.tensor(np.nanmean(x, axis=0), dtype=torch.float32)
        self.std = torch.tensor(np.nanstd(x, axis=0), dtype=torch.float32).clamp_min(eps)
    def encode(self, x):
        return (torch.as_tensor(x, dtype=torch.float32) - self.mean) / self.std
    def decode(self, z):
        return z * self.std.to(z.device) + self.mean.to(z.device)
    def to_dict(self):
        return {'mean': self.mean.cpu().numpy().tolist(), 'std': self.std.cpu().numpy().tolist()}

x_scaler = Standardizer(X_train)
y_scaler = Standardizer(Y_train)

def loader_for(X, Y, shuffle):
    Xz = x_scaler.encode(X)
    Yz = y_scaler.encode(Y)
    ds = TensorDataset(Xz, Yz)
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle, drop_last=False, num_workers=0)

train_loader = loader_for(X_train, Y_train, True)
val_loader = loader_for(X_val, Y_val, False)
test_loader = loader_for(X_test, Y_test, False)


In [8]:
class ResidualMLP(nn.Module):
    def __init__(self, in_dim, out_dim, hidden=256, depth=4, dropout=0.03):
        super().__init__()
        layers = []
        d = in_dim
        for _ in range(depth):
            layers += [nn.Linear(d, hidden), nn.LayerNorm(hidden), nn.SiLU(), nn.Dropout(dropout)]
            d = hidden
        layers.append(nn.Linear(d, out_dim))
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x)

model = ResidualMLP(X_train.shape[1], Y_train.shape[1], HIDDEN, DEPTH, DROPOUT).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS, eta_min=LR * 0.06)

target_index = {name: i for i, name in enumerate(TARGET_COLS)}
feature_index = {name: i for i, name in enumerate(FEATURE_COLS)}


In [9]:
def denorm_x(xz):
    return x_scaler.decode(xz)

def denorm_y(yz):
    return y_scaler.decode(yz)

def physics_losses(xz, pred_yz):
    x = denorm_x(xz)
    dy = denorm_y(pred_yz)
    dt = x[:, feature_index['dt_s']].clamp_min(1e-3)

    # relative_altitude_dot ~= -vel_down in NED coordinates. Use midpoint velocity.
    dx_h = dy[:, target_index['dx_relative_altitude_m']]
    vdown = x[:, feature_index['x_vel_down_m_s']]
    dvdown = dy[:, target_index['dx_vel_down_m_s']]
    vdown_mid = vdown + 0.5 * dvdown
    alt_res = dx_h / dt + vdown_mid
    alt_loss = torch.mean(alt_res ** 2)

    # small-angle yaw kinematic prior; deliberately weak for multicopter closed-loop data.
    dx_yaw_rad = torch.deg2rad(dy[:, target_index['dx_yaw_deg']])
    yaw_rate = x[:, feature_index['x_yaw_rate_rad_s']]
    yaw_res = dx_yaw_rad / dt - yaw_rate
    yaw_loss = torch.mean(yaw_res ** 2)

    rate_cols = ['dx_roll_rate_rad_s', 'dx_pitch_rate_rad_s', 'dx_yaw_rate_rad_s']
    rate_loss = torch.stack([dy[:, target_index[c]].pow(2).mean() for c in rate_cols]).mean()
    return alt_loss, yaw_loss, rate_loss

def run_epoch(loader, train_mode):
    model.train(train_mode)
    totals = {'loss': 0.0, 'data': 0.0, 'alt': 0.0, 'yaw': 0.0, 'rate': 0.0, 'n': 0}
    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)
        if train_mode:
            opt.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(train_mode):
            pred = model(xb)
            data_loss = torch.mean((pred - yb) ** 2)
            alt_loss, yaw_loss, rate_loss = physics_losses(xb, pred)
            loss = data_loss + ALT_KIN_WEIGHT * alt_loss + YAW_KIN_WEIGHT * yaw_loss + RATE_SMOOTH_WEIGHT * rate_loss
            if train_mode:
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                opt.step()
        bs = xb.shape[0]
        totals['loss'] += float(loss.detach().cpu()) * bs
        totals['data'] += float(data_loss.detach().cpu()) * bs
        totals['alt'] += float(alt_loss.detach().cpu()) * bs
        totals['yaw'] += float(yaw_loss.detach().cpu()) * bs
        totals['rate'] += float(rate_loss.detach().cpu()) * bs
        totals['n'] += bs
    return {k: v / max(totals['n'], 1) for k, v in totals.items() if k != 'n'}


In [10]:
best_val = float('inf')
best_state = None
history = []
t0 = time.time()
for epoch in range(1, EPOCHS + 1):
    tr = run_epoch(train_loader, True)
    va = run_epoch(val_loader, False)
    scheduler.step()
    row = {'epoch': epoch, **{f'train_{k}': v for k, v in tr.items()}, **{f'val_{k}': v for k, v in va.items()}, 'lr': scheduler.get_last_lr()[0]}
    history.append(row)
    if va['loss'] < best_val:
        best_val = va['loss']
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    if epoch == 1 or epoch % 20 == 0:
        print(f"epoch {epoch:04d} train={tr['loss']:.5f} val={va['loss']:.5f} data={va['data']:.5f} alt={va['alt']:.5f} yaw={va['yaw']:.5f}")

if best_state is not None:
    model.load_state_dict(best_state)
print('best val:', best_val, 'elapsed min:', round((time.time() - t0) / 60, 2))
history_df = pd.DataFrame(history)
display(history_df.tail())


epoch 0001 train=0.71853 val=0.46839 data=0.46801 alt=0.00454 yaw=0.00061
epoch 0020 train=0.09436 val=0.09813 data=0.09788 alt=0.00310 yaw=0.00019
epoch 0040 train=0.07743 val=0.08719 data=0.08698 alt=0.00254 yaw=0.00019
epoch 0060 train=0.06995 val=0.08252 data=0.08232 alt=0.00240 yaw=0.00020
epoch 0080 train=0.06413 val=0.07861 data=0.07839 alt=0.00267 yaw=0.00018
epoch 0100 train=0.06210 val=0.07676 data=0.07652 alt=0.00284 yaw=0.00018
epoch 0120 train=0.05840 val=0.07795 data=0.07774 alt=0.00260 yaw=0.00020
epoch 0140 train=0.05968 val=0.07520 data=0.07497 alt=0.00282 yaw=0.00018
epoch 0160 train=0.05623 val=0.07224 data=0.07202 alt=0.00264 yaw=0.00018
epoch 0180 train=0.05221 val=0.07313 data=0.07290 alt=0.00268 yaw=0.00020
epoch 0200 train=0.05169 val=0.07185 data=0.07161 alt=0.00282 yaw=0.00019
epoch 0220 train=0.05095 val=0.07247 data=0.07224 alt=0.00272 yaw=0.00018
epoch 0240 train=0.04969 val=0.07197 data=0.07175 alt=0.00271 yaw=0.00019
epoch 0260 train=0.04947 val=0.07282 d

,epoch,train_loss,train_data,train_alt,train_yaw,train_rate,val_loss,val_data,val_alt,val_yaw,val_rate,lr
255,256,0.049751,0.049396,0.004370,0.000115,0.000129,0.072721,0.072496,0.002723,0.000198,0.000177,0.000151
256,257,0.049318,0.048963,0.004382,0.000117,0.000131,0.072607,0.072383,0.002697,0.000197,0.000176,0.000151
257,258,0.048659,0.048307,0.004350,0.000113,0.000130,0.073061,0.072842,0.002645,0.000192,0.000176,0.000150
258,259,0.049356,0.049008,0.004293,0.000112,0.000130,0.073062,0.072838,0.002714,0.000197,0.000176,0.000150
259,260,0.049469,0.049124,0.004250,0.000114,0.000128,0.072815,0.072590,0.002718,0.000196,0.000178,0.000150


In [11]:
@torch.no_grad()
def predict_denorm(X):
    model.eval()
    Xz = x_scaler.encode(X).to(device)
    preds = []
    for i in range(0, len(Xz), 8192):
        yz = model(Xz[i:i+8192])
        preds.append(denorm_y(yz).cpu().numpy())
    return np.concatenate(preds, axis=0)

def metric_table(name, X, Y):
    P = predict_denorm(X)
    rows = []
    for i, col in enumerate(TARGET_COLS):
        err = P[:, i] - Y[:, i]
        rows.append({
            'split': name,
            'target': col.replace('dx_', ''),
            'rmse': float(np.sqrt(np.mean(err ** 2))),
            'mae': float(np.mean(np.abs(err))),
            'std_true': float(np.std(Y[:, i])),
        })
    return pd.DataFrame(rows)

metrics_df = pd.concat([
    metric_table('train', X_train, Y_train),
    metric_table('val', X_val, Y_val),
    metric_table('test', X_test, Y_test),
], ignore_index=True)
display(metrics_df)


,split,target,rmse,mae,std_true
0,train,relative_altitude_m,0.002197,0.001278,0.010789
1,train,vel_north_m_s,0.003585,0.001853,0.013895
2,train,vel_east_m_s,0.004228,0.002338,0.017091
3,train,vel_down_m_s,0.003615,0.001359,0.011886
4,train,roll_deg,0.015538,0.006342,0.143527
5,train,pitch_deg,0.017440,0.007564,0.167920
6,train,yaw_deg,0.013251,0.006310,0.113534
7,train,roll_rate_rad_s,0.003139,0.001339,0.013940
8,train,pitch_rate_rad_s,0.002652,0.001401,0.014171
9,train,yaw_rate_rad_s,0.000512,0.000293,0.004018


In [12]:
def rollout_segments(df, max_segments=24, horizon=80):
    # v7 keeps block_id-contiguous test windows, so rollout can evaluate real
    # consecutive trajectories instead of stitched random rows.
    out = []
    if 'block_id' in df.columns:
        group_cols = ['source_run', 'scenario', 'block_id'] if 'source_run' in df.columns else ['scenario', 'block_id']
    else:
        group_cols = ['source_run', 'scenario'] if 'source_run' in df.columns else ['scenario']
    ordered = df.sort_values([c for c in [*group_cols, 'sample_index'] if c in df.columns])
    for _, g in ordered.groupby(group_cols):
        g = g.sort_values('sample_index')
        sample_idx = g['sample_index'].to_numpy()
        start = 0
        for j in range(1, len(g) + 1):
            end_block = (j == len(g)) or (sample_idx[j] != sample_idx[j - 1] + 1)
            if end_block:
                block = g.iloc[start:j]
                if len(block) >= horizon:
                    out.append(block.iloc[:horizon].copy())
                    if len(out) >= max_segments:
                        return out
                start = j
    return out

@torch.no_grad()
def rollout_error(segment):
    cur = segment.iloc[0].copy()
    pred_states = []
    true_states = []
    for k in range(len(segment)):
        row = segment.iloc[k].copy()
        for c in STATE_COLS:
            row[f'x_{c}'] = cur[f'x_{c}']
        Xk, _ = make_features(pd.DataFrame([row]))
        dx = predict_denorm(Xk)[0]
        next_state = {}
        for i, c in enumerate(STATE_COLS):
            value = float(cur[f'x_{c}'] + dx[i])
            if c == 'yaw_deg':
                value = ((value + 180.0) % 360.0) - 180.0
            next_state[f'x_{c}'] = value
        pred_states.append([next_state[f'x_{c}'] for c in STATE_COLS])
        true_states.append([segment.iloc[k][f'x_next_{c}'] for c in STATE_COLS])
        cur = pd.Series(next_state)
    P = np.asarray(pred_states)
    T = np.asarray(true_states, dtype=np.float32)
    return {
        'scenario': str(segment.iloc[0]['scenario']),
        'ref_label': str(segment.iloc[0]['ref_label']),
        'horizon': len(segment),
        'alt_rmse_m': float(np.sqrt(np.mean((P[:, 0] - T[:, 0]) ** 2))),
        'vn_rmse_m_s': float(np.sqrt(np.mean((P[:, 1] - T[:, 1]) ** 2))),
        've_rmse_m_s': float(np.sqrt(np.mean((P[:, 2] - T[:, 2]) ** 2))),
        'yaw_rmse_deg': float(np.sqrt(np.mean((((P[:, 6] - T[:, 6] + 180) % 360) - 180) ** 2))),
    }

roll_rows = [rollout_error(seg) for seg in rollout_segments(test_df, max_segments=16, horizon=80)]
rollout_df = pd.DataFrame(roll_rows)
display(rollout_df)
print('rollout means:')
display(rollout_df.mean(numeric_only=True).to_frame('mean').T if len(rollout_df) else rollout_df)


,scenario,ref_label,horizon,alt_rmse_m,vn_rmse_m_s,ve_rmse_m_s,yaw_rmse_deg
0,C00_position_hold_25m,hold_25m,80,0.029277,0.013753,0.006607,0.040784
1,C00_position_hold_25m,hold_25m,80,0.019378,0.016258,0.026876,0.122836
2,C01_altitude_step_25_28_25m,return_25m,80,0.161087,0.023321,0.021283,0.065223
3,C01_altitude_step_25_28_25m,return_25m,80,0.056590,0.011461,0.019931,0.044987
4,C03_north_position_step_pm2m,recovery,80,0.056429,0.007718,0.011110,0.033329
5,C05_yaw_position_hold_pm20deg,yaw_zero_1,80,0.031457,0.016970,0.014861,0.178122
6,C09_mixed_position_sequence,recovery,80,0.015712,0.012912,0.057912,0.155024
7,C01_altitude_step_25_28_25m,pre_hold_25m,80,0.041496,0.014020,0.022414,0.147584
8,C02_altitude_step_25_22_25m,return_25m,80,0.065488,0.016167,0.017194,0.133930
9,C04_east_position_step_pm2m,east_plus_2m,80,0.024574,0.015271,0.033713,0.248313


rollout means:


,horizon,alt_rmse_m,vn_rmse_m_s,ve_rmse_m_s,yaw_rmse_deg
mean,80.0,0.053339,0.014065,0.020522,0.127598


In [13]:
if Path('/content/drive/MyDrive').exists():
    SAVE_ROOT = Path('/content/drive/MyDrive/Colab Result/PINN_MPC/px4_phase1_closed_loop_block_rollout_training_v15')
else:
    SAVE_ROOT = Path('/content/px4_phase1_closed_loop_block_rollout_training_v15')
RUN_STAMP = time.strftime('%Y%m%d_%H%M%S')
SAVE_DIR = SAVE_ROOT / RUN_STAMP
SAVE_DIR.mkdir(parents=True, exist_ok=True)

ckpt = {
    'revision': NOTEBOOK_REVISION,
    'dataset_dir': str(DATASET_DIR),
    'state_cols': STATE_COLS,
    'action_cols': ACTION_COLS,
    'target_cols': TARGET_COLS,
    'feature_cols': FEATURE_COLS,
    'model_config': {'hidden': HIDDEN, 'depth': DEPTH, 'dropout': DROPOUT},
    'model_state_dict': model.cpu().state_dict(),
    'x_scaler': x_scaler.to_dict(),
    'y_scaler': y_scaler.to_dict(),
    'history': history_df.to_dict(orient='records'),
}
torch.save(ckpt, SAVE_DIR / 'px4_closed_loop_block_rollout_dynamics_pinn_v15.pt')
history_df.to_csv(SAVE_DIR / 'training_history.csv', index=False)
metrics_df.to_csv(SAVE_DIR / 'one_step_metrics.csv', index=False)
rollout_df.to_csv(SAVE_DIR / 'rollout_metrics.csv', index=False)
print('saved:', SAVE_DIR)


saved: /content/drive/MyDrive/Colab Result/PINN_MPC/px4_phase1_closed_loop_block_rollout_training_v15/20260509_224053
